# When does a phenotype appear in a *C. elegans* lineage?

This notebook compares whole *C. elegans* lineages (from P0) with the
unordered tree edit distance (UTED), and uses the way the distance splits
over the subtrees to locate a phenotype in the lineage and in time.

The UTED is not normalised. With the `simple` style, each cell is one node
whose weight is its lifetime:

- matching two cells costs the difference of their lifetimes;
- a cell with no match costs its lifetime.

So the UTED of two subtrees is the distance between their two roots plus
the UTED of each pair of matched daughter subtrees, plus the cost of
unmatched daughter subtrees. If a phenotype affects one sublineage, the
distance between the two embryos is mainly carried by that subtree, much
more than its share of the cells.

In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

from lineagetree import LineageTreeManager

## Parameters

The embryos come from `all_celegans.ltm`, a `LineageTreeManager` holding 25
*C. elegans* lineages from the Bao lab: 5 wild-type embryos and 20 embryos
of their RNAi screen ([Du et al. 2015](https://doi.org/10.1016/j.devcel.2015.07.014)),
each with one gene knocked down. `EMBRYO_1` and `EMBRYO_2` are names of
trees in the manager, and `CELL` is the cell whose lineages are compared.

- **Speed:** embryos develop at different overall speeds (temperature,
  imaging). With `MATCH_SPEED = True`, the time resolution of embryo 1 is
  set so that its cell cycles match those of embryo 2 on average; times are
  then in frames of embryo 2.
- **Tracking:** lineages are not tracked equally far. Both lineages are
  compared from the birth of `CELL` until either one stops being fully
  tracked (its first leaf), so untracked cells do not count as
  differences.

In [2]:
MANAGER = Path("/Users/leo.guignard/tmp/all_celegans.ltm")
EMBRYO_1 = "Y110A7A.8_HYP_E1318"  # see the screen below
EMBRYO_2 = "info_ZD_RW10348_WT_20110126_2_s1_emb2_edited"  # wild type
CELL = "P0"

MATCH_SPEED = True

## Load the embryos

In [3]:
lTm = LineageTreeManager.load(MANAGER)
WILD_TYPES = [name for name, _ in lTm if name.startswith("info_ZD")]
for name, lT in lTm:
    tag = "WT" if name in WILD_TYPES else "  "
    print(f"{tag} {name:46} {len(lT.nodes):6} nodes, frames {lT.t_b}-{lT.t_e}")

WT info_ZD_RW10348_WT_20110427_3_s1_emb1_edited    20382 nodes, frames 0-189
WT info_ZD_RW10434_20110126_2_s4_emb2_edited       22347 nodes, frames 0-189
WT info_ZD_RW10348_WT_20110126_2_s1_emb3_edited    23484 nodes, frames 0-219
WT info_ZD_RW10348_WT_20110427_3_s1_emb2_edited    22657 nodes, frames 0-189
WT info_ZD_RW10348_WT_20110126_2_s1_emb2_edited    20880 nodes, frames 0-199
   REPO-1_PHA_E0966                                14712 nodes, frames 0-219
   SRC-1_HYP_E1121                                 21953 nodes, frames 0-209
   HIS-9_HYP_E0546                                 12579 nodes, frames 0-199
   F31C3.5_HYP_E0418                               14205 nodes, frames 0-239
   D1081.8_PHA_E0282                                6074 nodes, frames 0-153
   UBA-1_NEU_E1266                                 17411 nodes, frames 0-330
   GLD-2_HYP_E0491                                 16745 nodes, frames 0-299
   CDK-9_PHA_E0154                                 12043 nodes, frames 0-199

## Compare two lineages

`LineageComparison` aligns the lineages of `cell` in two embryos and splits
the UTED over the cells of embryo 1:

- each cell gets its own cost: the lifetime difference with its partner,
  or its lifetime if it has no partner;
- a cell of embryo 2 with no partner is attached to the closest ancestor of
  embryo 2 that has one;
- the **width** of a cell is its own cost, plus the cost of the cells of
  embryo 2 attached to it, plus the widths of its daughters. For a matched
  cell, this is the UTED of its subtree with its partner's subtree, which
  the comparison checks for every pair.

Every node of a chain carries the name of its cell, so a cell is identified
by the first node of its chain.

To match the speeds, each tree's `time_resolution` is set to 100 times its
speed factor: it is stored with one decimal, so this keeps the correction
precise to 0.1%. The manager counts durations in tenths of
`time_resolution`, hence the division by `10 * SCALE` to get frames. The
comparison keeps the resolutions it used, so that later comparisons, which
set them again, do not change it.

In [4]:
SCALE = 100


def cycles(lT):
    """Measure the cycle of every cell that divides.

    Parameters
    ----------
    lT : LineageTree
        The lineage of an embryo, with cell names as labels.

    Returns
    -------
    dict of {str: int}
        Maps the name of each cell that divides to the number of frames of
        its chain.
    """
    return {
        lT.labels[chain[0]]: len(chain)
        for chain in lT.all_chains
        if len(lT.successor[chain[-1]]) >= 2
    }


def first_nodes(lT):
    """Map the name of each cell to the first node of its chain.

    Parameters
    ----------
    lT : LineageTree
        The lineage of an embryo, with cell names as labels.

    Returns
    -------
    dict of {str: int}
        Maps each cell name to the first node of its chain.
    """
    return {lT.labels[chain[0]]: chain[0] for chain in lT.all_chains}


def tracked_until(lT, root):
    """Return the last frame at which the whole subtree of a node is tracked.

    Parameters
    ----------
    lT : LineageTree
        The lineage of an embryo.
    root : int
        The node spawning the subtree.

    Returns
    -------
    int
        The earliest time at which a branch of the subtree ends: up to this
        frame, every cell of the subtree is followed.
    """
    return min(
        lT.time[chain[-1]]
        for chain in lT.get_all_chains_of_subtree(root)
        if not lT.successor[chain[-1]]
    )


class LineageComparison:
    """Simple UTED between the lineages of a cell in two embryos, cell by cell.

    Matching two cells costs the difference of their lifetimes, and a cell
    without a partner costs its lifetime, in frames of embryo 2.

    Parameters
    ----------
    manager : LineageTreeManager
        The manager holding both embryos.
    name_1, name_2 : str
        The two embryos. The UTED is split over the cells of embryo 1.
    cell : str, default="P0"
        The cell whose lineages are compared.
    match_speed : bool, default=True
        Rescale embryo 1 so that its cell cycles match those of embryo 2 on
        average. Times are then in frames of embryo 2.

    Attributes
    ----------
    speed : float
        Median ratio of the cycle lengths of embryo 1 to those of embryo 2.
    resolution_1, resolution_2 : float
        Time resolutions given to the two trees, ``SCALE`` times the length
        of a frame of each embryo in frames of embryo 2.
    end_1, end_2 : int
        Last frame of each embryo in the comparison: both lineages are
        compared over the same duration, while they are fully tracked.
    partner : dict of {int: int}
        Maps each matched cell of embryo 1 to its partner in embryo 2.
    own : dict of {int: float}
        Cost of each cell of embryo 1: the difference of the lifetimes with
        its partner, or its lifetime if it has none.
    inserted : dict of {int: list of int}
        Maps a cell of embryo 1 to the cells of embryo 2 without a partner
        found below its partner.
    width : dict of {int: float}
        UTED carried by the subtree of each cell of embryo 1.
    n_cells : dict of {int: int}
        Number of cells of the subtree of each cell of embryo 1.
    """

    def __init__(self, manager, name_1, name_2, cell="P0", match_speed=True):
        """Align the two lineages and split their UTED over the cells of embryo 1.

        The parameters are described in the docstring of the class.
        """
        self.manager, self.name_1, self.name_2 = manager, name_1, name_2
        self.cell = cell
        self.lT1, self.lT2 = manager[name_1], manager[name_2]

        nodes_1, nodes_2 = first_nodes(self.lT1), first_nodes(self.lT2)
        self.root, self.root_2 = nodes_1[cell], nodes_2[cell]

        cycles_1, cycles_2 = cycles(self.lT1), cycles(self.lT2)
        self.speed = np.median(
            [cycles_1[c] / cycles_2[c] for c in cycles_1 if c in cycles_2]
        )
        self.resolution_1 = round(SCALE / self.speed, 1) if match_speed else SCALE
        self.resolution_2 = SCALE

        # same duration, in frames of embryo 2, while both are fully tracked
        duration = min(
            (tracked_until(self.lT1, self.root) - self.lT1.time[self.root])
            * self.resolution_1,
            (tracked_until(self.lT2, self.root_2) - self.lT2.time[self.root_2])
            * self.resolution_2,
        )
        self.end_1 = self.lT1.time[self.root] + int(duration / self.resolution_1)
        self.end_2 = self.lT2.time[self.root_2] + int(duration / self.resolution_2)

        self._set_resolutions()
        mapping = manager.labelled_mappings(
            self.root, name_1, self.root_2, name_2, self.end_1, self.end_2,
            style="simple",
        )
        self.partner = {nodes_1[a]: nodes_2[b] for a, b in mapping["matched"]}
        self._split()

    # time ------------------------------------------------------------------

    def _set_resolutions(self):
        """Give the two trees the time resolutions of the comparison.

        The manager reads the time resolutions from the trees, so they are
        set again before each comparison.
        """
        self.lT1.time_resolution = self.resolution_1
        self.lT2.time_resolution = self.resolution_2

    def time_1(self, node):
        """Return the time of a node of embryo 1, in frames of embryo 2.

        Parameters
        ----------
        node : int
            A node of embryo 1.

        Returns
        -------
        float
            Its time, rescaled to the frames of embryo 2.
        """
        return self.lT1.time[node] * self.resolution_1 / SCALE

    def time_2(self, node):
        """Return the time of a node of embryo 2.

        Parameters
        ----------
        node : int
            A node of embryo 2.

        Returns
        -------
        float
            Its time, in frames of embryo 2.
        """
        return self.lT2.time[node] * self.resolution_2 / SCALE

    def lifetime_1(self, cell):
        """Return how long a cell of embryo 1 lives within the compared window.

        Parameters
        ----------
        cell : int
            A node of a cell of embryo 1, usually its first one.

        Returns
        -------
        float
            The number of its frames up to ``end_1``, in frames of embryo 2.
        """
        chain = self.lT1.get_chain_of_node(cell)
        n = sum(self.lT1.time[node] <= self.end_1 for node in chain)
        return n * self.resolution_1 / SCALE

    def lifetime_2(self, cell):
        """Return how long a cell of embryo 2 lives within the compared window.

        Parameters
        ----------
        cell : int
            A node of a cell of embryo 2, usually its first one.

        Returns
        -------
        float
            The number of its frames up to ``end_2``.
        """
        chain = self.lT2.get_chain_of_node(cell)
        n = sum(self.lT2.time[node] <= self.end_2 for node in chain)
        return n * self.resolution_2 / SCALE

    def daughters_1(self, cell):
        """Return the daughters of a cell of embryo 1.

        Parameters
        ----------
        cell : int
            A node of a cell of embryo 1.

        Returns
        -------
        tuple of int
            The first nodes of its daughters, empty if the cell does not
            divide before ``end_1``.
        """
        last = self.lT1.get_chain_of_node(cell)[-1]
        return self.lT1.successor[last] if self.lT1.time[last] < self.end_1 else ()

    # costs -----------------------------------------------------------------

    def match_cost(self, cell_1, cell_2):
        """Return the cost of matching two cells.

        Parameters
        ----------
        cell_1 : int
            First node of a cell of embryo 1.
        cell_2 : int
            First node of a cell of embryo 2.

        Returns
        -------
        float
            The difference of their lifetimes, in frames of embryo 2.
        """
        return abs(self.lifetime_1(cell_1) - self.lifetime_2(cell_2))

    def uted(self, cell_1, cell_2):
        """Compute the UTED between two subtrees with the manager.

        Parameters
        ----------
        cell_1 : int
            First node of a cell of embryo 1.
        cell_2 : int
            First node of a cell of embryo 2.

        Returns
        -------
        float
            The unnormalised UTED (``simple`` style) between the subtrees of
            the two cells, up to ``end_1`` and ``end_2``, in frames of
            embryo 2.
        """
        self._set_resolutions()
        cost = self.manager.cross_lineage_edit_distance(
            cell_1, self.name_1, cell_2, self.name_2,
            self.end_1, self.end_2, style="simple", norm=None,
        )
        # the manager counts durations in tenths of `time_resolution`
        return cost / 10 / SCALE

    def _split(self):
        """Split the UTED over the cells of embryo 1, and check it.

        Fills ``own``, ``inserted``, ``width`` and ``n_cells``. The width of
        the subtree of a cell is its own cost, plus the lifetimes of the
        cells of embryo 2 inserted below its partner, plus the widths of its
        daughters.

        Raises
        ------
        AssertionError
            If the width of a matched cell differs from the UTED computed by
            the manager between its subtree and its partner's.
        """
        lT1, lT2 = self.lT1, self.lT2
        matched_2 = {c2: c1 for c1, c2 in self.partner.items()}
        chains_1 = [c[0] for c in lT1.get_all_chains_of_subtree(self.root, self.end_1)]
        chains_2 = [
            c[0] for c in lT2.get_all_chains_of_subtree(self.root_2, self.end_2)
        ]

        self.own = {
            c1: self.match_cost(c1, self.partner[c1])
            if c1 in self.partner
            else self.lifetime_1(c1)
            for c1 in chains_1
        }
        # cell of embryo 1 -> unmatched cells of embryo 2 below its partner
        self.inserted = {}
        for c2 in chains_2:
            if c2 not in matched_2:
                parent = lT2.get_chain_of_node(lT2.predecessor[c2][0])[0]
                while parent not in matched_2:
                    parent = lT2.get_chain_of_node(lT2.predecessor[parent][0])[0]
                self.inserted.setdefault(matched_2[parent], []).append(c2)

        self.width, self.n_cells = {}, {}

        def fill(c1):
            """Compute the width and the number of cells of a subtree.

            Parameters
            ----------
            c1 : int
                First node of a cell of embryo 1.

            Returns
            -------
            float
                The width of the subtree of ``c1``, also stored in ``width``.
            """
            kids = self.daughters_1(c1)
            self.width[c1] = (
                self.own[c1]
                + sum(self.lifetime_2(c2) for c2 in self.inserted.get(c1, []))
                + sum(fill(d) for d in kids)
            )
            self.n_cells[c1] = 1 + sum(self.n_cells[d] for d in kids)
            return self.width[c1]

        fill(self.root)
        for c1, c2 in self.partner.items():
            assert abs(self.width[c1] - self.uted(c1, c2)) < 1e-6, lT1.labels[c1]

    # summaries -------------------------------------------------------------

    @property
    def total(self):
        """Return the UTED between the two lineages.

        Returns
        -------
        float
            The width of the root, in frames of embryo 2.
        """
        return self.width[self.root]

    @property
    def cells(self):
        """Return the number of cells of the compared lineage of embryo 1.

        Returns
        -------
        int
            The number of cells of the subtree of the root, up to ``end_1``.
        """
        return self.n_cells[self.root]

    def name(self, cell):
        """Name a cell of embryo 1 together with its partner.

        Parameters
        ----------
        cell : int
            First node of a cell of embryo 1.

        Returns
        -------
        str
            ``"name ↔ partner"``, or ``"name (no match)"``.
        """
        if cell in self.partner:
            return f"{self.lT1.labels[cell]} ↔ {self.lT2.labels[self.partner[cell]]}"
        return f"{self.lT1.labels[cell]} (no match)"

    def most_loaded(self):
        """Find the subtree carrying the most distance compared with its size.

        Returns
        -------
        int
            The matched cell of embryo 1, other than the root, whose share of
            the UTED exceeds its share of the cells the most.
        float
            Its share of the UTED.
        float
            Its share of the cells.
        """
        best = max(
            (c for c in self.partner if c != self.root),
            key=lambda c: self.width[c] / self.total - self.n_cells[c] / self.cells,
        )
        return best, self.width[best] / self.total, self.n_cells[best] / self.cells

    def enrichment(self, cell):
        """Compare the share of the UTED of a subtree with its share of the cells.

        Parameters
        ----------
        cell : int
            First node of a cell of embryo 1.

        Returns
        -------
        float
            The share of the UTED carried by the subtree of ``cell``, divided
            by its share of the cells: 1 when the subtree carries what its
            size predicts.
        """
        return (self.width[cell] / self.total) / (self.n_cells[cell] / self.cells)

    def drill_down(self):
        """Print the path that carries the most distance, from the root.

        At each division, the daughter with the highest ``enrichment`` is
        followed, until a leaf, a cell without partner, or a subtree with
        less than 5% of the cells.
        """
        print(
            f"UTED between the {self.cell} lineages: "
            f"{self.total:.0f} frames, {self.cells} cells"
        )
        cell = self.root
        while True:
            kids = self.daughters_1(cell)
            if not kids:
                break
            followed = max(kids, key=self.enrichment)
            print(f"{self.lT1.labels[cell]} divides into:")
            for d in sorted(kids, key=self.enrichment, reverse=True):
                print(
                    f"  {self.name(d):18} {self.width[d] / self.total:4.0%} of the UTED, "
                    f"{self.n_cells[d] / self.cells:4.0%} of the cells "
                    f"({self.enrichment(d):.1f}×)"
                    + ("   <-" if d == followed else "")
                )
            cell = followed
            if self.n_cells[cell] / self.cells < 0.05 or cell not in self.partner:
                break

## Where is the distance carried?

For each embryo, compare its lineage of `CELL` with the wild type
`EMBRYO_2`, and find the subtree that carries the largest share of the
distance compared with its share of the cells. Between wild types, every
subtree carries about its share of the cells: that is the background.

In [5]:
print(
    f"{'embryo':44} {'speed':>5} {'UTED':>6}   most loaded subtree: "
    "share of the UTED (share of the cells)"
)
for name, _ in lTm:
    if name == EMBRYO_2:
        continue
    comparison = LineageComparison(lTm, name, EMBRYO_2, CELL, match_speed=MATCH_SPEED)
    best, share, cell_share = comparison.most_loaded()
    tag = "WT" if name in WILD_TYPES else "  "
    print(
        f"{tag} {name:41} {comparison.speed:5.2f} {comparison.total:6.0f}   "
        f"{comparison.lT1.labels[best]:8} {share:4.0%} ({cell_share:4.0%})"
    )

embryo                                       speed   UTED   most loaded subtree: share of the UTED (share of the cells)


WT info_ZD_RW10348_WT_20110427_3_s1_emb1_edited  0.95   1414   ABprp     13% (  9%)


WT info_ZD_RW10434_20110126_2_s4_emb2_edited  0.98   1134   ABalp     13% ( 10%)


WT info_ZD_RW10348_WT_20110126_2_s1_emb3_edited  1.00   1124   ABp       43% ( 37%)


WT info_ZD_RW10348_WT_20110427_3_s1_emb2_edited  0.96   1766   P1        31% ( 24%)


...

   TBX-33_NEU_E1252                           0.92    553   ABp       53% ( 30%)


   C01A2.5_PHA_E0052                          1.08   1140   E         22% (  5%)


   F35F11.1_NEU_E0426                         1.48   3518   P2        15% ( 12%)
   TAT-5_HYP_E1242                            1.17     37   P1        61% ( 47%)


   CAMT-1_HYP_E0119                           1.26   1582   ABp       58% ( 35%)


   ETR-1_PHA_E0385                            1.12   1186   ABa       38% ( 33%)


Between wild types, no subtree carries much more than its share of the
cells. **Y110A7A.8_HYP_E1318** develops at the speed of the wild type, and
its distance is concentrated in the EMS lineage, then in E, the intestine
lineage (SKP-1_NEU_E1082 and C01A2.5_PHA_E0052 show the same, less
cleanly).

## Following the distance down the lineage

Starting from `CELL`, at each division follow the daughter that carries
the most distance compared with its number of cells (its share of the UTED
divided by its share of the cells, 1 being the expected value).

In [6]:
comparison = LineageComparison(lTm, EMBRYO_1, EMBRYO_2, CELL, match_speed=MATCH_SPEED)
comparison.drill_down()

UTED between the P0 lineages: 1474 frames, 353 cells
P0 divides into:
  P1 ↔ P1             51% of the UTED,  28% of the cells (1.8×)   <-
  AB ↔ AB             48% of the UTED,  72% of the cells (0.7×)
P1 divides into:
  EMS ↔ EMS           40% of the UTED,  16% of the cells (2.5×)   <-
  P2 ↔ P2             11% of the UTED,  12% of the cells (1.0×)
EMS divides into:
  E ↔ E               26% of the UTED,   7% of the cells (3.7×)   <-
  MS ↔ MS             14% of the UTED,   9% of the cells (1.6×)
E divides into:
  Ea ↔ Ep             12% of the UTED,   3% of the cells (3.9×)   <-
  Ep (no match)       13% of the UTED,   4% of the cells (3.6×)


The distance goes down P1, then EMS, then E: E has 7% of the cells
but carries about a quarter of the distance.

## When does it start?

Cell cycle lengths along the E lineage and of its neighbours, in frames,
against the mean ± standard deviation of the 5 wild types.

In [7]:
def wild_type_cycles(manager, wild_types):
    """Describe the cycle of each cell in the wild types.

    Parameters
    ----------
    manager : LineageTreeManager
        The manager holding the embryos.
    wild_types : list of str
        Names of the wild-type embryos.

    Returns
    -------
    dict of {str: tuple of (float, float)}
        Maps each cell measured in at least three wild types to the mean and
        the standard deviation of its cycle length, in frames.
    """
    all_cycles = [cycles(manager[w]) for w in wild_types]
    reference = {}
    for c in set().union(*all_cycles):
        values = [wc[c] for wc in all_cycles if c in wc]
        if len(values) >= 3:
            reference[c] = (np.mean(values), np.std(values, ddof=1))
    return reference


def cycle_table(lT, cell_names, reference, speed=1):
    """Print the cycles of some cells of an embryo against the wild type.

    Cycles more than 4 standard deviations away from the wild type are
    flagged.

    Parameters
    ----------
    lT : LineageTree
        The lineage of the embryo.
    cell_names : list of str
        The cells to print.
    reference : dict of {str: tuple of (float, float)}
        Mean and standard deviation of the wild-type cycles, from
        ``wild_type_cycles``.
    speed : float, default=1
        Pace of the embryo relative to the wild types: its cycles are
        divided by it.
    """
    observed, born = cycles(lT), first_nodes(lT)
    print(f"{'cell':6} {'born':>5} {'wild type':>11} {lT.name:>20}")
    for c in cell_names:
        mean, sd = reference[c]
        value = observed[c] / speed
        z = (value - mean) / max(sd, 1)
        print(
            f"{c:6} {lT.time[born[c]]:5} {mean:6.1f} ± {sd:<3.1f} {value:12.1f}"
            + (f"   {z:+.0f} SD" if abs(z) > 4 else "")
        )


cycle_table(
    comparison.lT1,
    ["EMS", "MS", "MSa", "MSp", "E", "Ea", "Ep", "Eal", "Ear", "Epl", "Epr", "P2", "C"],
    wild_type_cycles(lTm, WILD_TYPES),
    speed=comparison.speed,
)

cell    born   wild type  Y110A7A.8_HYP_E1318
EMS       16   12.6 ± 0.5         12.5
MS        28   14.4 ± 1.1         14.6
MSa       42   17.0 ± 0.7         15.7
MSp       42   16.8 ± 0.8         15.7
E         28   15.4 ± 1.1         14.6
Ea        42   33.6 ± 2.4         18.8   -6 SD
Ep        42   34.4 ± 2.5         19.8   -6 SD
Eal       60   36.6 ± 2.4         25.0   -5 SD
Ear       60   36.6 ± 2.5         25.0   -5 SD
Epl       61   38.6 ± 2.4         24.0   -6 SD
Epr       61   38.2 ± 3.2         23.0   -5 SD
P2        16   15.6 ± 1.1         15.7
C         31   15.6 ± 0.9         15.7


Up to E, the cycles are those of the wild type. The phenotype starts
with the **daughters of E**: Ea and Ep, born around frame 42, normally have
a long cycle (about 34 frames), during which the gut precursors move inside
the embryo at gastrulation. In Y110A7A.8_HYP_E1318 they divide after 18–19
frames, and their daughters keep dividing early.

## UTED through the subtrees

Each matched cell of embryo 1 is a bar whose **width** is the UTED of its
subtree with its partner's subtree, in frames, and whose **height** is its
lifetime in embryo 1. The bars of its daughters are drawn below it;
together they are narrower than their parent by the distance between the
two roots, which is the gap left on each side. The **colour** is the share
of the total UTED carried by the subtree: the subtrees that carry a large
part of the distance stay dark down the tree, and show where the two
lineages deviate. **Hatched** bars are
cells with no match: `///` for cells of embryo 1, and `xxx` for cells of
embryo 2 (drawn with their lifetime in embryo 2).

Hover over a rectangle to see its cells and costs; drag to zoom,
double-click to zoom out.

In [8]:
class ComparisonFigure:
    """Draw a `LineageComparison`: one bar per cell of embryo 1.

    The width of a bar is the UTED of its subtree with its partner's
    subtree, its height the lifetime of the cell in embryo 1. Daughters are
    drawn below their mother, centred, the root cost being the gap left on
    each side. `share`, `enrichment` and `mother` draw the figure with
    different colours.

    Parameters
    ----------
    comparison : LineageComparison
        The comparison to draw.

    Attributes
    ----------
    c : LineageComparison
        The comparison.
    rects : list of dict
        The rectangles to draw, with their position, size, colour values
        and hover text.
    """

    def __init__(self, comparison):
        """Lay out the rectangles of the comparison.

        The parameters are described in the docstring of the class.
        """
        self.c = comparison
        self.rects = []
        self._layout(comparison.root, 0, None)

    def _layout(self, cell, x0, mother_width):
        """Add the rectangle of a cell, then those of its descendants.

        The rectangles of the cells of embryo 2 without partner below the
        partner of the cell come first, then the daughters, side by side.

        Parameters
        ----------
        cell : int
            First node of a cell of embryo 1.
        x0 : float
            Left side of the rectangle.
        mother_width : float or None
            Width of the mother's rectangle, None for the root.
        """
        c = self.c
        top, life = c.time_1(cell), c.lifetime_1(cell)
        share = c.width[cell] / c.total
        cell_share = c.n_cells[cell] / c.cells
        rect = {
            "x0": x0,
            "width": c.width[cell],
            "top": top,
            "height": life,
            "share": share,
            # 1 when the subtree carries the share of the distance its size predicts
            "enrichment": share / cell_share,
            # the root has no mother: neutral
            "share_of_mother": c.width[cell] / mother_width if mother_width else 0.5,
            "root": mother_width is None,
        }
        hover = (
            f"<b>{c.name(cell)}</b><br>"
            f"subtree UTED: {c.width[cell]:.1f} frames<br>"
            f"share of the total: {100 * share:.1f}% for {100 * cell_share:.1f}% "
            f"of the cells ({rect['enrichment']:.2f}× expected)<br>"
        )
        if mother_width:
            hover += f"share of the mother: {100 * rect['share_of_mother']:.0f}%<br>"
        if cell in c.partner:
            partner = c.partner[cell]
            hover += (
                f"root cost: {c.own[cell]:.1f}<br>"
                f"lifetime: {life:.1f} vs {c.lifetime_2(partner):.1f} frames<br>"
                f"born at: {top:.1f} vs {c.time_2(partner):.1f}<br>"
                f"cells in the subtree: {c.n_cells[cell]}"
            )
            kind = "matched"
        else:
            hover += f"born at: {top:.1f}"
            kind = "only in embryo 1"
        rect |= {"kind": kind, "name": c.name(cell), "hover": hover}
        self.rects.append(rect)

        x = x0 + c.own[cell] / 2
        for cell_2 in c.inserted.get(cell, []):
            cost = c.lifetime_2(cell_2)
            self.rects.append(
                {
                    "x0": x,
                    "width": cost,
                    "top": top + life,
                    "height": cost,
                    "kind": "only in embryo 2",
                    "name": c.lT2.labels[cell_2],
                    "hover": (
                        f"<b>{c.lT2.labels[cell_2]}</b>: no match in embryo 1<br>"
                        f"cost: {cost:.1f} frames (its lifetime)<br>"
                        f"born at: {c.time_2(cell_2):.1f} in embryo 2"
                    ),
                }
            )
            x += cost
        for d in sorted(c.daughters_1(cell), key=c.lT1.labels.get):
            self._layout(d, x, c.width[cell])
            x += c.width[d]

    def _bars(self, kind, label=None, **marker):
        """Make one bar trace with the rectangles of a kind.

        Parameters
        ----------
        kind : str
            ``"matched"``, ``"only in embryo 1"`` or ``"only in embryo 2"``.
        label : Callable, optional
            Maps a rectangle to the text written in it, after its name; only
            wide rectangles are labelled.
        **marker
            Passed to the marker of the trace (colour, colour scale...).

        Returns
        -------
        plotly.graph_objects.Bar
            The trace.
        """
        rs = [r for r in self.rects if r["kind"] == kind]
        label_width = 0.06 * self.c.total  # label only the wide bars
        return go.Bar(
            x=[r["x0"] + r["width"] / 2 for r in rs],
            width=[r["width"] for r in rs],
            base=[r["top"] for r in rs],
            y=[r["height"] for r in rs],
            text=[
                f"{r['name']}<br>{label(r)}" if label and r["width"] > label_width else ""
                for r in rs
            ],
            textposition="inside",
            insidetextanchor="middle",
            hovertext=[r["hover"] for r in rs],
            hoverinfo="text",
            name=kind,
            marker=dict(line=dict(width=0.5, color="white"), **marker),
        )

    def plot(self, colour, label, subtitle, **colour_scale):
        """Draw the figure.

        Parameters
        ----------
        colour : Callable
            Maps a rectangle of a matched cell to its colour value.
        label : Callable
            Maps a rectangle to the text written in it.
        subtitle : str
            Explains the colours, below the title.
        **colour_scale
            Colour scale of the matched cells, passed to their marker.

        Returns
        -------
        plotly.graph_objects.Figure
            The figure.
        """
        c = self.c
        matched = [r for r in self.rects if r["kind"] == "matched"]
        fig = go.Figure(
            [
                self._bars(
                    "matched", label, color=[colour(r) for r in matched], **colour_scale
                ),
                self._bars("only in embryo 1", color="lightgrey", pattern=dict(shape="/")),
                self._bars("only in embryo 2", color="lightgrey", pattern=dict(shape="x")),
            ]
        )
        fig.update_layout(
            title=(
                f"{c.cell}: {c.name_1} (embryo 1) vs {c.name_2} (embryo 2)<br>"
                f"<sup>UTED {c.total:.0f} frames. {subtitle}</sup>"
            ),
            xaxis=dict(title="UTED (frames)", range=[0, c.total]),
            yaxis=dict(
                title="time in embryo 1 (frames of embryo 2)", autorange="reversed"
            ),
            barmode="overlay",
            bargap=0,
            uniformtext=dict(minsize=6, mode="hide"),
            legend=dict(orientation="h", y=-0.12),
            height=750,
            plot_bgcolor="white",
        )
        return fig

    def share(self):
        """Draw the figure coloured by the share of the total UTED of each subtree.

        Returns
        -------
        plotly.graph_objects.Figure
            The figure.
        """
        return self.plot(
            colour=lambda r: 100 * r["share"],
            label=lambda r: f"{100 * r['share']:.0f}%",
            subtitle="colour: share of the total UTED carried by the subtree",
            colorscale="Magma",
            reversescale=True,
            cmin=0,
            cmax=100,
            colorbar=dict(title="share of the<br>total UTED", ticksuffix="%"),
        )

    def enrichment(self):
        """Draw the figure coloured by the share of the UTED over the share of the cells.

        Returns
        -------
        plotly.graph_objects.Figure
            The figure.
        """
        return self.plot(
            colour=lambda r: float(np.clip(np.log2(max(r["enrichment"], 1e-3)), -3, 3)),
            label=lambda r: f"{r['enrichment']:.1f}×",
            subtitle="colour: share of the total UTED ÷ share of the cells (1 = expected)",
            colorscale="RdBu",
            reversescale=True,
            cmin=-3,
            cmax=3,
            colorbar=dict(
                title="UTED share ÷<br>cell share",
                tickvals=[-3, -2, -1, 0, 1, 2, 3],
                ticktext=["1/8", "1/4", "1/2", "1", "2", "4", "8"],
            ),
        )

    def mother(self):
        """Draw the figure coloured by the share of the mother's UTED of each subtree.

        Returns
        -------
        plotly.graph_objects.Figure
            The figure.
        """
        return self.plot(
            colour=lambda r: 100 * r["share_of_mother"],
            label=lambda r: "–" if r["root"] else f"{100 * r['share_of_mother']:.0f}%",
            subtitle="colour: share of the mother's UTED carried by the subtree",
            colorscale="RdBu",
            reversescale=True,
            cmin=0,
            cmax=100,
            cmid=50,
            colorbar=dict(title="share of the<br>mother's UTED", ticksuffix="%"),
        )


figure = ComparisonFigure(comparison)
figure.share().show()

The dark path goes from P0 down P1, EMS and E: E ↔ E carries a quarter
of the distance for 7% of the cells, and its daughters still carry 12–13%
each, while the subtrees of the same generation elsewhere carry a few
percent at most. The deviation is in the E lineage, from the daughters of E
on (the hover gives the lifetimes of the two partners: equal for E, much
shorter in embryo 1 for its daughters).

Because the E cells of embryo 1 divide faster, the two E subtrees no longer
have the same shape within the compared window. The alignment then pairs
Ea of embryo 1 with Ep of the wild type, and leaves Ep of embryo 1 without a
partner (hatched `///`), while some of its descendants are matched again.
The UTED ignores cell names, so such pairings happen whenever they cost
less; the hover gives both names.

### Normalised by the number of cells

A subtree with a tenth of the cells is expected to carry about a tenth of
the distance. Dividing its share of the total UTED by its share of the
cells gives 1 for a subtree that differs as much as the rest, more than 1
(red) for a subtree that carries more than its size predicts, and less than
1 (blue) for one that carries less. The scale is logarithmic, from 1/8 to
8.

In [9]:
figure.enrichment().show()

The lineage splits into blue and red at the first division: AB carries
0.6–0.8× its expected share, all the way down, while P1 carries 1.8×, EMS
2.5× and E 3.7×. Below E, the excess grows (Ea ↔ Ep 3.9×, up to about 8×
for the granddaughters of E): the deviation is in the E lineage. MS is
mildly red (1.5–1.8×), and P2 and C are at about 1×.

### Share of the mother's distance

Each bar is coloured by the share of its mother's UTED that it carries.
When a cell divides and its two daughters differ as much from the wild type,
each carries about half of the distance below their mother (a bit less, as
the mother's own cost is part of it): white. A daughter that takes most of
it is red, its sister blue. Following the red bars from the root gives the
path to the deviation, and the level where the colours turn back to white
is where it stops being carried by one branch.

In [10]:
figure.mother().show()

P0 splits evenly between AB (48%) and P1 (51%), although AB has most
of the cells. Then the distance takes one branch: EMS carries 78% of P1's
distance, and E 65% of EMS's. Below E, the colours turn back to white: Ea
and Ep share E's distance evenly. So the deviation enters the lineage at E,
and is spread over all its descendants rather than one of them, as the
cycle lengths showed: both daughters of E divide early.

Deep in the tree, subtrees are small and a difference of a few frames can
split a mother's distance unevenly, so strong colours at the bottom of the
figure mean little.